<a href="https://colab.research.google.com/github/data4class/Teaching/blob/main/Transformer_demo_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Transformer: Explanation with Demo

WHAT IS A TRANSFORMER?

A Transformer is a neural network architecture that processes sequences by using
attention mechanisms to understand relationships between all elements simultaneously.

Key Innovation: Instead of processing sequences sequentially (word-by-word),
transformers process all positions in parallel and use attention to determine
which parts of the input are most relevant to each other.

MAIN COMPONENTS:

1. EMBEDDINGS
   Converts discrete tokens (words) into continuous vector representations.
   Similar concepts are mapped to similar vectors in the embedding space.

2. POSITIONAL ENCODING
   Adds position information to embeddings since the model processes all
   tokens simultaneously. Allows the model to use word order information.

3. ATTENTION MECHANISM
   Core innovation that computes relationships between all pairs of positions.
   For each position, determines how much to focus on every other position.
   Allows modeling of long-range dependencies efficiently.

4. MULTI-HEAD ATTENTION
   Runs multiple attention operations in parallel, each potentially capturing
   different types of relationships (syntactic, semantic, etc.).

5. FEED-FORWARD NETWORKS
   Position-wise fully connected layers that process the attention outputs.
   Same transformation applied independently to each position.

6. LAYER NORMALIZATION & RESIDUAL CONNECTIONS
   Stabilizes training and enables deeper architectures.
   Residual connections allow gradient flow through many layers.

ARCHITECTURE FLOW:

Input Tokens → Embeddings + Positional Encoding →
→ [Multi-Head Attention → Add & Norm → Feed Forward → Add & Norm] × N layers
→ Output

ADVANTAGES:

1. Parallelization: All positions processed simultaneously (efficient training)
2. Long-range dependencies: Direct connections between distant positions
3. Flexibility: Same architecture works for various sequence tasks
4. Scalability: Performance improves with model size and data

ARCHITECTURE VARIANTS:

1. Encoder-only (BERT): Bidirectional understanding, good for classification
2. Decoder-only (GPT): Autoregressive generation, good for text generation
3. Encoder-Decoder (T5): Sequence-to-sequence, good for translation

APPLICATIONS:

- Language modeling (GPT series)
- Machine translation (many modern systems)
- Text classification (BERT for search, sentiment analysis)
- Question answering
- Summarization
- Code generation



REFERENCES AND RESOURCES:

1. Original Paper:
   "Attention Is All You Need" - Vaswani et al., 2017
   https://arxiv.org/abs/1706.03762

2. Excellent Tutorials:
   - "The Illustrated Transformer" by Jay Alammar
     https://jalammar.github.io/illustrated-transformer/
   
   - "The Annotated Transformer" by Harvard NLP
     https://nlp.seas.harvard.edu/2018/04/03/attention.html
   
   - "Transformers from Scratch" by Peter Bloem
     https://peterbloem.nl/blog/transformers

3. Video Tutorials:
   - "Attention is All You Need" - Yannic Kilcher
     https://www.youtube.com/watch?v=iDulhoQ2pro
   
   - Stanford CS224N Lecture on Transformers
     https://www.youtube.com/watch?v=5vcj8kSwBCY

4. Interactive Resources:
   - "Visualizing Attention in Transformer Models" - Jesse Vig
     https://github.com/jessevig/bertviz
   
   - Hugging Face Transformers Course
     https://huggingface.co/course/chapter1/1

5. Books:
   - "Natural Language Processing with Transformers" by Tunstall, von Werra, and Wolf
   - "Speech and Language Processing" by Jurafsky and Martin (Chapter on Transformers)


## FINANCIAL SENTIMENT ANALYSIS USING BERT

OVERVIEW:
This code implements a sentiment analysis system for financial news using BERT
(Bidirectional Encoder Representations from Transformers). It classifies financial
text into three categories: positive, neutral, or negative sentiment.

WHAT THIS CODE DOES:

1. DATA LOADING
   - Loads financial news data from Google Drive (CSV format)
   - Expected columns: 'Sentence' (text) and 'Sentiment' (label)
   - Splits data into training (80%) and validation (20%) sets

2. DATA PREPROCESSING
   - Uses BERT tokenizer to convert text into numerical tokens
   - Adds special tokens ([CLS], [SEP]) required by BERT
   - Pads/truncates sequences to fixed length (64 tokens)
   - Creates PyTorch Dataset and DataLoader for batch processing

3. MODEL SETUP
   - Uses pre-trained BERT-base-uncased model (110M parameters)
   - Adds classification head with 3 output classes
   - Transfers model to GPU if available for faster training

4. TRAINING
   - Fine-tunes BERT on financial sentiment data for 3 epochs
   - Uses AdamW optimizer with learning rate 2e-5
   - Batch size: 16 samples per iteration
   - Computes training loss and validation accuracy after each epoch

5. EVALUATION & PREDICTION
   - Tracks validation loss and accuracy during training
   - Provides predict_sentiment() function for new text
   - Returns sentiment labels with confidence scores

KEY COMPONENTS:

- FinancialNewsDataset: Custom PyTorch Dataset class for data handling
- BERT Model: Pre-trained transformer for contextual understanding
- AdamW Optimizer: Weight decay regularization for better generalization
- Softmax: Converts model outputs to probability distribution

EXPECTED INPUT FORMAT (CSV):
   Sentence                                    | Sentiment
   -------------------------------------------|----------
   "Stock prices surge after earnings report" | positive
   "Company faces regulatory challenges"      | negative
   "Market shows mixed signals"               | neutral

OUTPUT:
   - Training/validation metrics printed after each epoch
   - Sentiment predictions for new headlines

In [6]:
# Import necessary libraries
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Check if CUDA is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load and preprocess the data
from google.colab import drive
drive.mount('/content/drive')


Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Replace with your actual file path
file_path = '/content/drive/My Drive/sentiment_data.csv'

# Read the CSV file
df = pd.read_csv(file_path)
print(df.head())


                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3  According to the Finnish-Russian Chamber of Co...   neutral
4  The Swedish buyout firm has sold its remaining...   neutral


In [4]:

# Split the data into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Sentence'].tolist(),
    df['Sentiment'].tolist(),
    test_size=0.2,
    random_state=41
)

# Define label encoding
label_encoding = {'positive': 0, 'neutral': 1, 'negative': 2}


In [8]:
# Create a custom dataset class
class FinancialNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = [label_encoding[label] for label in labels]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create train and validation datasets
max_len = 64
train_dataset = FinancialNewsDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset = FinancialNewsDataset(val_texts, val_labels, tokenizer, max_len)

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Initialize the model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
model.to(device)

# Set up the optimizer - FIXED: Import from torch.optim
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Training loop
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_loss = 0
    correct_predictions = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

    val_accuracy = correct_predictions.double() / len(val_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Training Loss: {train_loss/len(train_loader):.4f}")
    print(f"Validation Loss: {val_loss/len(val_loader):.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}\n")

# Function to predict sentiment for new texts
def predict_sentiment(texts):
    model.eval()
    encoded_texts = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    input_ids = encoded_texts['input_ids'].to(device)
    attention_mask = encoded_texts['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

    return [list(label_encoding.keys())[pred] for pred in preds]

# Test the model with new financial news headlines
test_headlines = [
    "Investors optimistic about new tech IPO",
    "Major bank announces significant job cuts",
    "Market remains stable despite geopolitical tensions",
    "The Enforcement Directorate (ED) has arrested the CFO of industrialist Anil Ambani's group company Reliance Power in a money laundering case linked to issuance of an alleged fake bank guarantee of Rs 68 crore, official sources said on Saturday."
]

predictions = predict_sentiment(test_headlines)

for headline, sentiment in zip(test_headlines, predictions):
    print(f"Headline: {headline}")
    print(f"Predicted sentiment: {sentiment}\n")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3
Training Loss: 0.6240
Validation Loss: 0.4761
Validation Accuracy: 0.7938

Epoch 2/3
Training Loss: 0.3487
Validation Loss: 0.4069
Validation Accuracy: 0.8127

Epoch 3/3
Training Loss: 0.2314
Validation Loss: 0.4297
Validation Accuracy: 0.8144

Headline: Investors optimistic about new tech IPO
Predicted sentiment: positive

Headline: Major bank announces significant job cuts
Predicted sentiment: negative

Headline: Market remains stable despite geopolitical tensions
Predicted sentiment: positive

Headline: The Enforcement Directorate (ED) has arrested the CFO of industrialist Anil Ambani's group company Reliance Power in a money laundering case linked to issuance of an alleged fake bank guarantee of Rs 68 crore, official sources said on Saturday.
Predicted sentiment: neutral

